In [ ]:
# --- imports 
import polars as pl
import pandas as pd
import numpy as np
import os
import requests
import logging

# logging configuration
logging.basicConfig(level=logging.INFO)

In [ ]:
# --- constants 

USED_NODE: str = "thin002"
T_START:   str = "2026-08-11T00:00:00.000000Z"
T_END:     str = "2026-08-11T23:59:59.000000Z"
DATA_DIR:  str = "/Users/isacpasianotto/Desktop/workdirectoy/paper-enegy/data-analysis/data"
SENSOR_FILE: str = DATA_DIR + "/ipmi_sensors.arrow"
POWER_FILE:  str = DATA_DIR + "/ipmi_power.arrow"
QUESTDB_ENDPOINT: str = "https://timeseriesdb.dev.rd.areasciencepark.it"


QUERY_SENSORS = f"""
SELECT *
FROM ipmi_sensor
WHERE host LIKE '{USED_NODE}%'
AND timestamp >= '{T_START}'
AND timestamp <= '{T_END}'
ORDER BY timestamp ASC;
"""

QUERY_POWER = f"""
SELECT *
FROM ipmi_power
WHERE host LIKE '{USED_NODE}%'
AND timestamp >= '{T_START}'
AND timestamp <= '{T_END}'
ORDER BY timestamp ASC;
"""

In [ ]:
def get_table_data(filename: str, query: str) -> pl.DataFrame:
    """
    Check if `filename` exists, if so try to read it. 
    Otherwise, query the QuestDB via REST API and save the result to `filename`.
    """
    
    if os.path.exists(filename):
        try:
            logging.info(f"Reading data from {filename}")
            return pl.read_ipc(filename)
        except Exception as e:
            print(f"Error reading {filename}: {e}")
            return pl.DataFrame()  # return empty DataFrame on error

    # --- query a QuestDB via REST API ---
    logging.info(f"Querying QuestDB and saving to {filename}")
    
    response = requests.get(
        f"{QUESTDB_ENDPOINT}/exec",
        params={"query": query},
        verify=False  # Disable SSL verification for self-signed certificates
    )
    response.raise_for_status()
    payload = response.json()

    columns = [c["name"] for c in payload["columns"]]
    rows = payload["dataset"]

    df = pl.DataFrame(rows, schema=columns, orient="row")

    # --- convert columns to appropriate types ---
    logging.debug(f"Converting columns to appropriate types for {filename}")
    df = df.with_columns(
        pl.col("timestamp").str.strptime(
            pl.Datetime, "%Y-%m-%dT%H:%M:%S%.fZ", strict=False
        ),
        pl.col("value").cast(pl.Float64),
    )
    for col in ("name", "type", "unit"):
        if col in df.columns:
            df = df.with_columns(pl.col(col).cast(pl.Utf8))

    if "psu_id" in df.columns:
        df = df.with_columns(pl.col("psu_id").cast(pl.Int64, strict=False))

    os.makedirs(os.path.dirname(filename), exist_ok=True)
    df.write_ipc(filename)

    return df

In [ ]:
power_df: pl.DataFrame = get_table_data(POWER_FILE, QUERY_POWER)
sensor_df: pl.DataFrame = get_table_data(SENSOR_FILE, QUERY_SENSORS)

## Data-cleaning 

***Step 1:*** Reshape the 2 dataframe keeping only the data of interests, and merge them into a single one

In [ ]:
power_reshaped: pl.DataFrame = (
    power_df
    .filter(
        (pl.col("unit") == "W") &
        (pl.col("psu_id") == 0)
    )
    .select(
        "timestamp",
        pl.col("value").alias("power_W")
    )
    .drop_nulls()
)

In [ ]:
# print(sensor_df.head())

sensors_filtered = pl.sql("""
    SELECT host, name, value, timestamp
    FROM sensor_df
    WHERE unit IN ('C', 'RPM')
""").collect()

sensors_reshaped: pl.DataFrame = sensors_filtered.pivot(
    on="name",
    index=["host", "timestamp"],
    values="value",
    ).drop("host").drop_nulls().sort("timestamp")

Merge the two Dataframe on the `timestamp` column.

In [ ]:
# NOTE --> `join_asof` merge the df on a column with the closest value, while merge 
#           checks for exact matches. This is useful with time series :)

merged_df: pl.DataFrame = power_reshaped.sort("timestamp").join_asof(
    sensors_reshaped.sort("timestamp"),
    on="timestamp",
    strategy="nearest", # or "backward" or "forward"
    tolerance="500m",
).drop_nulls()

In [ ]:
print(merged_df.head())